## Job Failure Scraper — Runbook

- Use widgets to configure the time window, workspace host, page limit, log table, and token source.
- Ensure `job_failure_scraper.py` and `args.py` are available in this Repo so imports work.


In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

# Create widgets (run this cell first, then populate values)
# Host is auto-detected from the current workspace; no host widget needed

dbutils.widgets.removeAll()

dbutils.widgets.text("start", "2025-09-23 08:00:00", "Start time (UTC) e.g. 2025-09-23 08:00:00")
dbutils.widgets.text("end", "2025-09-23 10:59:59", "End time (UTC) e.g. 2025-09-23 10:59:59")
dbutils.widgets.text("limit", "26", "API Page Limit (max 26) e.g. 26")
dbutils.widgets.text("log_table", "your_catalog.your_schema.job_failure_log", "Log table (catalog.schema.table) e.g. catalog.schema.job_failure_log")
dbutils.widgets.text("token", "", "PAT (optional; leave blank to use secret)")
dbutils.widgets.text("secret_scope", "", "Secret Scope (optional)")
dbutils.widgets.text("secret_key", "", "Secret Key (optional)")


In [ ]:
import job_failure_scraper as jfs
import requests

# Read widgets
start = dbutils.widgets.get("start").strip()
end = dbutils.widgets.get("end").strip()
limit = dbutils.widgets.get("limit").strip()
log_table = dbutils.widgets.get("log_table").strip()
user_token = dbutils.widgets.get("token").strip()
secret_scope = dbutils.widgets.get("secret_scope").strip()
secret_key = dbutils.widgets.get("secret_key").strip()

# Auto-detect host from current workspace
host = "https://" + spark.conf.get("spark.databricks.workspaceUrl")

# Validate a token with a lightweight API call
def validate_token(host_url: str, token_value: str):
  try:
    resp = requests.get(
      f"{host_url}/api/2.2/jobs/runs/list",
      headers={"Authorization": f"Bearer {token_value}"},
      params={"limit": 1},
      timeout=30,
    )
    if resp.status_code < 400:
      return True, None
    return False, f"HTTP {resp.status_code}: {resp.text}"
  except Exception as e:
    return False, str(e)

# Resolve token: try PAT widget first; if empty or invalid, try secret scope/key
errors = []
pat = None
if user_token:
  ok, err = validate_token(host, user_token)
  if ok:
    pat = user_token
  else:
    errors.append(f"PAT (widget) validation failed: {err}")
else:
  errors.append("Token widget empty")

if pat is None:
  if secret_scope and secret_key:
    try:
      secret_pat = dbutils.secrets.get(scope=secret_scope, key=secret_key)
      ok, err = validate_token(host, secret_pat)
      if ok:
        pat = secret_pat
      else:
        errors.append(f"PAT (secret) validation failed: {err}")
    except Exception as e:
      errors.append(f"Secret retrieval failed: {e}")
  else:
    if not secret_scope:
      errors.append("Secret scope widget empty")
    if not secret_key:
      errors.append("Secret key widget empty")

if pat is None:
  raise ValueError("No valid PAT available. Reasons: " + "; ".join(errors))

# Build args
args = [
  "--start", start,
  "--end", end,
  "--host", host,
  "--token", pat,
  "--limit", limit,
]
if log_table:
  args += ["--log-table", log_table]

jfs.main(args)
